In [54]:
# Create data file with 20k samples using uniform distributions
# over T, P, qc, and qv multiplier
%run ../SatAdj_GenerateSamples.py -f samples20k.csv -N 20000 -v

Summary stats
	mean delta T: 0.15579233233328502
	variance in delta T: 1.6882659768247874
	max delta T: 12.353239687037672
	min delta T: -44.089002793617
	min |delta T|: 4.792582330992445e-09 

	mean delta qv: -6.25973591315139e-05
	variance in delta qv: 2.7255912928035866e-07
	max delta qv: 2.7255912928035866e-07
	min delta qv: -0.004963531706251738
	min |delta qv|: 1.925661143574627e-12 

	mean delta qc: 6.25973591315139e-05
	variance in delta qc: 2.7255912928035866e-07
	max delta qc: 0.004963531706251738
	min delta qc: -0.017714961322475298
	min |delta qc|: 1.9256610195306223e-12


In [55]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, QuantileTransformer

In [56]:
df = pd.read_csv('samples20k.csv')

In [57]:
X = df[['T_in', 'pres_in', 'qv_in', 'qc_in']].values
Y = df[['T_out', 'qv_out', 'qc_out']].values

In [58]:
scaler_x = QuantileTransformer()
scaler_y = QuantileTransformer()

In [59]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30)

X_train_tensor = torch.tensor(X_train, dtype=torch.float64)
X_test_tensor = torch.tensor(X_test, dtype=torch.float64)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float64)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float64)

In [60]:
X_train_scaled = torch.tensor(scaler_x.fit_transform(X_train_tensor), dtype=torch.float64)
X_test_scaled = torch.tensor(scaler_x.fit_transform(X_test_tensor), dtype=torch.float64)
Y_train_scaled = torch.tensor(scaler_y.fit_transform(Y_train_tensor), dtype=torch.float64)
Y_test_scaled = torch.tensor(scaler_y.fit_transform(Y_test_tensor), dtype=torch.float64)

In [61]:
X_train_scaled

tensor([[0.0530, 0.7243, 0.0598, 0.9785],
        [0.7680, 0.5865, 0.8192, 0.6255],
        [0.1733, 0.9730, 0.0170, 0.0863],
        ...,
        [0.2934, 0.5805, 0.3337, 0.5811],
        [0.5036, 0.3412, 0.5331, 0.1031],
        [0.3444, 0.5464, 0.3319, 0.4836]], dtype=torch.float64)

In [62]:
class NeuralNet(nn.Module):
    def __init__(self, n_in, n_out, width):
        super(NeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(n_in, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, n_out)
        )

    def forward(self, x):
        return self.model(x)

In [63]:
n_in = 4
n_out = 3
width = 32
m_tol = 1e-6

net = NeuralNet(n_in, n_out, width)
net.double()
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters())

In [64]:
epochs = 5000
for epoch in range(epochs):
    net.train()
    
    pred = net(X_train_scaled)
    loss = criterion(pred, Y_train_scaled)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3e}")

    if (loss.item() < m_tol) :
        print(f"Breaking at Epoch {epoch}, Loss: {loss.item():.3e}")
        break

Epoch 0, Loss: 2.454e-01
Epoch 100, Loss: 7.778e-03
Epoch 200, Loss: 2.758e-03
Epoch 300, Loss: 2.193e-03
Epoch 400, Loss: 2.002e-03
Epoch 500, Loss: 1.922e-03
Epoch 600, Loss: 1.851e-03
Epoch 700, Loss: 1.805e-03
Epoch 800, Loss: 1.773e-03
Epoch 900, Loss: 1.747e-03
Epoch 1000, Loss: 1.722e-03
Epoch 1100, Loss: 1.699e-03
Epoch 1200, Loss: 1.679e-03
Epoch 1300, Loss: 1.657e-03
Epoch 1400, Loss: 1.635e-03
Epoch 1500, Loss: 1.612e-03
Epoch 1600, Loss: 1.590e-03
Epoch 1700, Loss: 1.568e-03
Epoch 1800, Loss: 1.545e-03
Epoch 1900, Loss: 1.520e-03
Epoch 2000, Loss: 1.492e-03
Epoch 2100, Loss: 1.465e-03
Epoch 2200, Loss: 1.438e-03
Epoch 2300, Loss: 1.414e-03
Epoch 2400, Loss: 1.390e-03
Epoch 2500, Loss: 1.367e-03
Epoch 2600, Loss: 1.339e-03
Epoch 2700, Loss: 1.316e-03
Epoch 2800, Loss: 1.287e-03
Epoch 2900, Loss: 1.262e-03
Epoch 3000, Loss: 1.233e-03
Epoch 3100, Loss: 1.207e-03
Epoch 3200, Loss: 1.185e-03
Epoch 3300, Loss: 1.164e-03
Epoch 3400, Loss: 1.144e-03
Epoch 3500, Loss: 1.125e-03
Epoc

In [65]:
net.eval()
with torch.no_grad():
    Y_pred = net(X_test_scaled)
    mse_loss = nn.MSELoss()
    
    loss_T = mse_loss(Y_pred[0], Y_test_scaled[0])
    print("Scaled Test MSE T:", loss_T.item())

    loss_qv = mse_loss(Y_pred[1], Y_test_scaled[1])
    print("Scaled Test MSE qv:", loss_qv.item())

    loss_qc = mse_loss(Y_pred[2], Y_test_scaled[2])
    print("Scaled Test MSE qc:", loss_qc.item())

Scaled Test MSE T: 2.060544670565797e-05
Scaled Test MSE qv: 0.00015719164897470318
Scaled Test MSE qc: 2.2188361090173653e-05


In [66]:
Y_pred = torch.tensor(scaler_y.inverse_transform(Y_pred), dtype=torch.float64)

loss_T = mse_loss(Y_pred[0], Y_test_tensor[0])
print("Unscaled Test MSE T:", loss_T.item())

loss_qv = mse_loss(Y_pred[1], Y_test_tensor[1])
print("Unscaled Test MSE qv: ", loss_qv.item())

loss_qc = mse_loss(Y_pred[2], Y_test_tensor[2])
print("Unscaled Test MSE qc:", loss_qc.item())

Unscaled Test MSE T: 0.00047076191324744846
Unscaled Test MSE qv:  0.005961259342811192
Unscaled Test MSE qc: 0.00106110740059222
